# Module 1: Spark Basics Refresher

**Objective**: Get comfortable with SparkSession, DataFrames, and basic operations.

**Time**: ~1-2 hours

---

## What You'll Learn
1. Creating and configuring a SparkSession
2. Reading data from CSV files
3. Basic DataFrame operations (select, filter, show)
4. Schema inference vs explicit schema
5. Writing data to Parquet format

## 1. SparkSession Setup

SparkSession is the entry point to Spark. It replaces the old SparkContext and SQLContext.

In [1]:
# Use project config (sets HADOOP_HOME and optimized settings)
import sys
sys.path.insert(0, "..")
from configs.spark_config import get_spark_session

# Create SparkSession using project config
spark = get_spark_session("Module01-SparkBasics")

# Verify
print(f"Spark Version: {spark.version}")
print(f"App Name: {spark.sparkContext.appName}")
print(f"\n🌐 Spark UI available at: http://localhost:4040")

Spark Version: 3.4.1
App Name: Module01-SparkBasics

🌐 Spark UI available at: http://localhost:4040


### 💡 Key Concepts

| Component | Description |
|-----------|-------------|
| `appName` | Name shown in Spark UI |
| `master("local[*]")` | Use all CPU cores locally |
| `spark.sql.shuffle.partitions` | Partitions after shuffle (default 200 is too high for local) |
| `spark.driver.memory` | Memory for the driver JVM |

## 2. Reading Data

Let's read the synthetic banking data we generated.

In [2]:
from pathlib import Path

# Define paths
DATA_PATH = Path("../data/raw")

# Check if data exists
if not (DATA_PATH / "customers.csv").exists():
    print("⚠️ Data not found! Run the data generator first:")
    print("   python src/data_generator.py")
else:
    print("✅ Data files found!")

✅ Data files found!


In [3]:
# Read CSV with schema inference
customers_df = spark.read.csv(
    str(DATA_PATH / "customers.csv"),
    header=True,
    inferSchema=True
)

# Show first 5 rows
customers_df.show(5, truncate=False)

+-----------+-------------+------------------+---------------+-------------+-----------------+----------+-------------+------+-----------+
|customer_id|name         |email             |phone          |segment      |registration_date|kyc_status|date_of_birth|gender|nationality|
+-----------+-------------+------------------+---------------+-------------+-----------------+----------+-------------+------+-----------+
|CUST000001 |An Vũ        |johnle@example.net|+84 81 9600133 |Mass Affluent|2021-03-09       |Verified  |1985-02-27   |M     |Vietnamese |
|CUST000002 |Vân Phạm     |cpham@example.net |00 2654 2351   |Mass         |2024-11-26       |Verified  |1966-09-08   |M     |Vietnamese |
|CUST000003 |Kim Trần     |jane94@example.org|+84-78-161 8495|Affluent     |2021-07-13       |Verified  |1979-04-12   |M     |Vietnamese |
|CUST000004 |Cô Ngọc Phạm |john31@example.com|07 5255 3419   |Mass         |2019-02-02       |Pending   |1995-09-17   |M     |Vietnamese |
|CUST000005 |Chị Bảo Dương|

In [4]:
# Check the inferred schema
customers_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- registration_date: date (nullable = true)
 |-- kyc_status: string (nullable = true)
 |-- date_of_birth: date (nullable = true)
 |-- gender: string (nullable = true)
 |-- nationality: string (nullable = true)



In [5]:
# Basic statistics
print(f"Total customers: {customers_df.count():,}")
print(f"Columns: {len(customers_df.columns)}")
print(f"Partitions: {customers_df.rdd.getNumPartitions()}")

Total customers: 10,000
Columns: 10
Partitions: 1


## 3. Explicit Schema Definition

Schema inference is convenient but slow for large files. Explicit schemas are faster and more reliable.

In [6]:
from pyspark.sql.types import (
    StructType, StructField, StringType, DateType, DoubleType
)

# Define explicit schema for accounts
accounts_schema = StructType([
    StructField("account_id", StringType(), False),
    StructField("customer_id", StringType(), False),
    StructField("branch_id", StringType(), False),
    StructField("account_type", StringType(), True),
    StructField("balance", DoubleType(), True),
    StructField("currency", StringType(), True),
    StructField("status", StringType(), True),
    StructField("opened_date", StringType(), True),
    StructField("last_activity_date", StringType(), True)
])

# Read with explicit schema (faster!)
accounts_df = spark.read.csv(
    str(DATA_PATH / "accounts.csv"),
    header=True,
    schema=accounts_schema
)

accounts_df.printSchema()
accounts_df.show(5)

root
 |-- account_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- branch_id: string (nullable = true)
 |-- account_type: string (nullable = true)
 |-- balance: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- status: string (nullable = true)
 |-- opened_date: string (nullable = true)
 |-- last_activity_date: string (nullable = true)

+----------+-----------+---------+------------+--------------+--------+-------+-----------+------------------+
|account_id|customer_id|branch_id|account_type|       balance|currency| status|opened_date|last_activity_date|
+----------+-----------+---------+------------+--------------+--------+-------+-----------+------------------+
|ACCT000001| CUST000001| BR000052|  Investment| 8.253134778E7|     VND|Dormant| 2024-03-31|        2025-05-11|
|ACCT000002| CUST000002| BR000032|    Checking|    7693641.71|     VND| Active| 2022-09-13|        2024-12-22|
|ACCT000003| CUST000003| BR000068|     Savings|2.7682417839

## 4. Basic DataFrame Operations

### 4.1 Selecting Columns

In [7]:
from pyspark.sql.functions import col

# Method 1: String column names
customers_df.select("customer_id", "name", "segment").show(5)

# Method 2: Using col() function
customers_df.select(col("customer_id"), col("name"), col("segment")).show(5)

# Method 3: Using DataFrame reference
customers_df.select(customers_df.customer_id, customers_df.name).show(5)

+-----------+-------------+-------------+
|customer_id|         name|      segment|
+-----------+-------------+-------------+
| CUST000001|        An Vũ|Mass Affluent|
| CUST000002|     Vân Phạm|         Mass|
| CUST000003|     Kim Trần|     Affluent|
| CUST000004| Cô Ngọc Phạm|         Mass|
| CUST000005|Chị Bảo Dương|     Affluent|
+-----------+-------------+-------------+
only showing top 5 rows

+-----------+-------------+-------------+
|customer_id|         name|      segment|
+-----------+-------------+-------------+
| CUST000001|        An Vũ|Mass Affluent|
| CUST000002|     Vân Phạm|         Mass|
| CUST000003|     Kim Trần|     Affluent|
| CUST000004| Cô Ngọc Phạm|         Mass|
| CUST000005|Chị Bảo Dương|     Affluent|
+-----------+-------------+-------------+
only showing top 5 rows

+-----------+-------------+
|customer_id|         name|
+-----------+-------------+
| CUST000001|        An Vũ|
| CUST000002|     Vân Phạm|
| CUST000003|     Kim Trần|
| CUST000004| Cô Ngọc Phạm

### 4.2 Filtering Rows

In [8]:
# Filter high-net-worth customers
hnw_customers = customers_df.filter(col("segment").isin(["HNW", "UHNW"]))
print(f"HNW/UHNW Customers: {hnw_customers.count():,}")
hnw_customers.show(5)

HNW/UHNW Customers: 997
+-----------+---------------+-------------------+--------------+-------+-----------------+----------+-------------+------+-----------+
|customer_id|           name|              email|         phone|segment|registration_date|kyc_status|date_of_birth|gender|nationality|
+-----------+---------------+-------------------+--------------+-------+-----------------+----------+-------------+------+-----------+
| CUST000007|  Khoa Hữu Đặng|johnmai@example.com|+84 01 2269166|    HNW|       2017-08-27|   Pending|   1995-11-18|     M| Vietnamese|
| CUST000014|    Tú Hải Phạm| janele@example.com|(04) 7317 8108|    HNW|       2018-02-17|  Verified|   1982-11-07|     F| Vietnamese|
| CUST000037|Quý cô Hải Trần| janevu@example.com|(09) 9124 1904|    HNW|       2024-12-30|  Verified|   1996-07-23|     F| Vietnamese|
| CUST000047|    Nhật Mai Vũ|    kvu@example.net|+84 91 3341232|    HNW|       2018-05-08|  Verified|   1998-04-13|     F| Vietnamese|
| CUST000052|      Nhật Đặng|  

In [9]:
# Multiple conditions
verified_hnw = customers_df.filter(
    (col("segment") == "HNW") & 
    (col("kyc_status") == "Verified")
)
print(f"Verified HNW: {verified_hnw.count():,}")

Verified HNW: 609


### 4.3 Sorting

In [10]:
# Sort accounts by balance (descending)
accounts_df.orderBy(col("balance").desc()).show(10)

+----------+-----------+---------+------------+-----------------+--------+------+-----------+------------------+
|account_id|customer_id|branch_id|account_type|          balance|currency|status|opened_date|last_activity_date|
+----------+-----------+---------+------------+-----------------+--------+------+-----------+------------------+
|ACCT012501| CUST007597| BR000037|    Checking|4.998257286334E10|     VND|Active| 2018-04-08|        2025-09-13|
|ACCT006852| CUST004143| BR000091|    Checking|4.986269900792E10|     VND|Active| 2018-12-14|        2025-08-29|
|ACCT005656| CUST003433| BR000061|  Investment|4.966902166654E10|     VND|Active| 2021-12-26|        2025-01-05|
|ACCT005918| CUST003590| BR000007|  Investment|4.961725988391E10|     VND|Active| 2023-04-01|        2025-11-05|
|ACCT014442| CUST008796| BR000023|      Credit|4.935819548725E10|     VND|Active| 2024-11-09|        2025-01-25|
|ACCT004609| CUST002794| BR000097|Term Deposit|4.912108934692E10|     VND|Active| 2023-01-22|   

### 4.4 Distinct Values

In [11]:
# Get unique segments
customers_df.select("segment").distinct().show()

# Count by segment
customers_df.groupBy("segment").count().orderBy("count", ascending=False).show()

+-------------+
|      segment|
+-------------+
|Mass Affluent|
|         Mass|
|          HNW|
|     Affluent|
|         UHNW|
+-------------+

+-------------+-----+
|      segment|count|
+-------------+-----+
|         Mass| 5077|
|Mass Affluent| 2384|
|     Affluent| 1542|
|          HNW|  715|
|         UHNW|  282|
+-------------+-----+



## 5. Lazy Evaluation Demo

**Key Concept**: Transformations (filter, select, join) are LAZY - they don't execute until an ACTION is called.

In [12]:
# These are all TRANSFORMATIONS (lazy, no execution yet)
result = customers_df \
    .filter(col("segment") == "Affluent") \
    .select("customer_id", "name", "segment") \
    .orderBy("name")

print("Transformations defined (nothing executed yet)")
print(f"Query plan ready: {result.isLocal}")

Transformations defined (nothing executed yet)
Query plan ready: <bound method DataFrame.isLocal of DataFrame[customer_id: string, name: string, segment: string]>


In [13]:
# View the execution plan
result.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [name#18 ASC NULLS FIRST], true, 0
   +- Exchange rangepartitioning(name#18 ASC NULLS FIRST, 8), ENSURE_REQUIREMENTS, [plan_id=309]
      +- Filter (isnotnull(segment#21) AND (segment#21 = Affluent))
         +- FileScan csv [customer_id#17,name#18,segment#21] Batched: false, DataFilters: [isnotnull(segment#21), (segment#21 = Affluent)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/D:/huyng/Documents/Obsidian/MyObsidian/_System/Projects/Spark-li..., PartitionFilters: [], PushedFilters: [IsNotNull(segment), EqualTo(segment,Affluent)], ReadSchema: struct<customer_id:string,name:string,segment:string>




In [14]:
# NOW it executes (show() is an ACTION)
result.show(5)

+-----------+------+--------+
|customer_id|  name| segment|
+-----------+------+--------+
| CUST003701|An Bùi|Affluent|
| CUST002510|An Bùi|Affluent|
| CUST009425|An Bùi|Affluent|
| CUST006634|An Bùi|Affluent|
| CUST006920|An Bùi|Affluent|
+-----------+------+--------+
only showing top 5 rows



### Actions vs Transformations

| Transformations (Lazy) | Actions (Trigger Execution) |
|------------------------|-----------------------------|
| `select()` | `show()` |
| `filter()` | `count()` |
| `groupBy()` | `collect()` |
| `join()` | `take(n)` |
| `orderBy()` | `write.*` |

## 6. Writing Data to Parquet

Parquet is a columnar format - faster and smaller than CSV.

In [15]:
OUTPUT_PATH = Path("../data/processed")
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

# Write customers to Parquet
customers_df.write \
    .mode("overwrite") \
    .parquet(str(OUTPUT_PATH / "customers_parquet"))

print("✅ Customers saved to Parquet!")

✅ Customers saved to Parquet!


In [16]:
# Read back and verify
customers_parquet = spark.read.parquet(str(OUTPUT_PATH / "customers_parquet"))
print(f"Rows: {customers_parquet.count():,}")
customers_parquet.show(3)

Rows: 10,000
+-----------+--------+------------------+---------------+-------------+-----------------+----------+-------------+------+-----------+
|customer_id|    name|             email|          phone|      segment|registration_date|kyc_status|date_of_birth|gender|nationality|
+-----------+--------+------------------+---------------+-------------+-----------------+----------+-------------+------+-----------+
| CUST000001|   An Vũ|johnle@example.net| +84 81 9600133|Mass Affluent|       2021-03-09|  Verified|   1985-02-27|     M| Vietnamese|
| CUST000002|Vân Phạm| cpham@example.net|   00 2654 2351|         Mass|       2024-11-26|  Verified|   1966-09-08|     M| Vietnamese|
| CUST000003|Kim Trần|jane94@example.org|+84-78-161 8495|     Affluent|       2021-07-13|  Verified|   1979-04-12|     M| Vietnamese|
+-----------+--------+------------------+---------------+-------------+-----------------+----------+-------------+------+-----------+
only showing top 3 rows



## 7. Spark SQL

You can also use SQL syntax directly!

In [17]:
# Register DataFrame as a temp view
customers_df.createOrReplaceTempView("customers")
accounts_df.createOrReplaceTempView("accounts")

# Now use SQL!
spark.sql("""
    SELECT segment, COUNT(*) as customer_count
    FROM customers
    GROUP BY segment
    ORDER BY customer_count DESC
""").show()

+-------------+--------------+
|      segment|customer_count|
+-------------+--------------+
|         Mass|          5077|
|Mass Affluent|          2384|
|     Affluent|          1542|
|          HNW|           715|
|         UHNW|           282|
+-------------+--------------+



In [18]:
# Join customers and accounts using SQL
spark.sql("""
    SELECT 
        c.name,
        c.segment,
        a.account_type,
        a.balance
    FROM customers c
    JOIN accounts a ON c.customer_id = a.customer_id
    ORDER BY a.balance DESC
    LIMIT 10
""").show()

+---------------+-------+------------+-----------------+
|           name|segment|account_type|          balance|
+---------------+-------+------------+-----------------+
|        Hồng Vũ|   UHNW|    Checking|4.998257286334E10|
|     Mai Nguyễn|   UHNW|    Checking|4.986269900792E10|
|Thành Phú Hoàng|   UHNW|  Investment|4.966902166654E10|
|Cô Dương Nguyễn|   UHNW|  Investment|4.961725988391E10|
|    Duyên Dương|   UHNW|      Credit|4.935819548725E10|
| Tùng Hữu Dương|   UHNW|Term Deposit|4.912108934692E10|
|  Phương Nguyễn|   UHNW|      Credit|4.889213870333E10|
|   Ông Dũng Bùi|   UHNW|      Credit|4.886555309968E10|
|  Chị Hà Nguyễn|   UHNW|  Investment|4.879707801365E10|
|  Bà Nhật Hoàng|   UHNW|     Savings|4.858168539152E10|
+---------------+-------+------------+-----------------+



## ✅ Practice Exercises

Complete these exercises to solidify your understanding:

In [19]:
# Exercise 1: Read transactions.csv and show the schema
# Your code here:
transactions_df = spark.read.csv(
    str(DATA_PATH / "transactions.csv"),
    header=True,
    inferSchema=True
)
transactions_df.printSchema()

root
 |-- txn_id: string (nullable = true)
 |-- account_id: string (nullable = true)
 |-- txn_datetime: timestamp (nullable = true)
 |-- txn_type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- merchant_category: string (nullable = true)
 |-- status: string (nullable = true)
 |-- reference: string (nullable = true)
 |-- description: string (nullable = true)



In [20]:
import os

# Get file size in bytes
file_path = str(DATA_PATH / "transactions.csv")
file_size_bytes = os.path.getsize(file_path)

# Get Spark DataFrame estimated size in bytes
# This uses the logical plan statistics
df_size_bytes = transactions_df._jdf.queryExecution().optimizedPlan().stats().sizeInBytes()

print(f"File size (CSV): {file_size_bytes / (1024**2):.2f} MB")
print(f"DataFrame size (Memory): {df_size_bytes / (1024**2):.2f} MB")
print(f"Comparison ratio: {df_size_bytes / file_size_bytes:.2f}x")


File size (CSV): 579.78 MB
DataFrame size (Memory): 579.78 MB
Comparison ratio: 1.00x


In [21]:
transactions_df.limit(1000).show()
# .collect() is slow because it is an action that:
# 1. Triggers the execution of all preceding transformations in the Spark DAG.
# 2. Transfers all data from distributed executors to the single driver node over the network.
# 3. Requires the driver to have enough memory to store the entire dataset, which can lead to OOM errors.
# 4. Involves significant serialization and deserialization overhead.

# .take(1000) is faster because it:
# 1. Only transfers a small subset of data (1000 rows) to the driver node.
# 2. Avoids the need for full dataset transfer and memory constraints.
# 3. Is less resource-intensive in terms of network and memory usage.
# 4. Is still a transformation (lazy), so it doesn't trigger execution until an action is called.


+---------+----------+-------------------+------------+--------------+--------+----------------+-----------------+---------+------------+--------------------+
|   txn_id|account_id|       txn_datetime|    txn_type|        amount|currency|         channel|merchant_category|   status|   reference|         description|
+---------+----------+-------------------+------------+--------------+--------+----------------+-----------------+---------+------------+--------------------+
|TXN000001|ACCT006097|2025-01-01 10:10:03|     Deposit| 6.851589517E7|     VND|      Mobile App|             null|Completed|REF926342336| Deposit transaction|
|TXN000002|ACCT015768|2025-03-12 11:53:16|  Withdrawal| 4.432876995E7|     VND|          Branch|             null|Completed|REF257178572|Withdrawal transa...|
|TXN000003|ACCT012175|2025-09-29 03:09:38|     Payment| 3.055445133E7|     VND|      Mobile App|       Healthcare|Completed|REF332870623| Payment transaction|
|TXN000004|ACCT010179|2025-05-31 17:57:26|    

In [22]:
transactions_df.take(5)

[Row(txn_id='TXN000001', account_id='ACCT006097', txn_datetime=datetime.datetime(2025, 1, 1, 10, 10, 3), txn_type='Deposit', amount=68515895.17, currency='VND', channel='Mobile App', merchant_category=None, status='Completed', reference='REF926342336', description='Deposit transaction'),
 Row(txn_id='TXN000002', account_id='ACCT015768', txn_datetime=datetime.datetime(2025, 3, 12, 11, 53, 16), txn_type='Withdrawal', amount=44328769.95, currency='VND', channel='Branch', merchant_category=None, status='Completed', reference='REF257178572', description='Withdrawal transaction'),
 Row(txn_id='TXN000003', account_id='ACCT012175', txn_datetime=datetime.datetime(2025, 9, 29, 3, 9, 38), txn_type='Payment', amount=30554451.33, currency='VND', channel='Mobile App', merchant_category='Healthcare', status='Completed', reference='REF332870623', description='Payment transaction'),
 Row(txn_id='TXN000004', account_id='ACCT010179', txn_datetime=datetime.datetime(2025, 5, 31, 17, 57, 26), txn_type='Paym

In [23]:
# Exercise 2: Count total transactions
# Your code here:
CountTotalTransactions = transactions_df.count()
print(f"Total number of transactions: {CountTotalTransactions}")

Total number of transactions: 5000000


In [24]:
# Exercise 3: Find all "Failed" transactions
# Your code here:
FailedTransactions = transactions_df.filter("status='Failed'")
FailedTransactions.show()


+---------+----------+-------------------+------------+--------------+--------+----------------+-----------------+------+------------+--------------------+
|   txn_id|account_id|       txn_datetime|    txn_type|        amount|currency|         channel|merchant_category|status|   reference|         description|
+---------+----------+-------------------+------------+--------------+--------+----------------+-----------------+------+------------+--------------------+
|TXN000026|ACCT013027|2025-01-12 17:18:40|         Fee|      77236.81|     VND|      Mobile App|             null|Failed|REF654235744|     Fee transaction|
|TXN000031|ACCT010285|2025-12-12 01:41:11|     Payment| 2.359198875E7|     VND|          Branch|           Others|Failed|REF999331285| Payment transaction|
|TXN000097|ACCT008713|2025-07-17 01:30:33|  Withdrawal| 8.650074239E7|     VND|             ATM|             null|Failed|REF231726057|Withdrawal transa...|
|TXN000274|ACCT015189|2025-07-07 11:50:05|    Interest|1.5304931

In [25]:
# Exercise 4: Group transactions by channel and count
# Your code here:
TransactionsGroupedByChannel = transactions_df.groupBy("channel").count()
TransactionsGroupedByChannel.show()

+----------------+------+
|         channel| count|
+----------------+------+
|      Mobile App|832776|
|             POS|832721|
|          Branch|834369|
|Internet Banking|833933|
|             API|832662|
|             ATM|833539|
+----------------+------+



In [26]:
# Exercise 5: Write transactions to Parquet, partitioned by txn_type
# Hint: Use .partitionBy("txn_type")
# Your code here:
OUTPUT_PATH = Path("../data/processed")
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

# Write customers to Parquet
transactions_df.repartition(100).write \
    .mode("overwrite") \
    .partitionBy("txn_type") \
    .parquet(str(OUTPUT_PATH / "transactions_parquet"))

print("✅ Customers saved to Parquet!")

✅ Customers saved to Parquet!


## 🎯 Key Takeaways

1. **SparkSession** is your entry point - configure memory and partitions for local dev
2. **Explicit schemas** are faster than inferSchema for large files
3. **Lazy evaluation** means transformations only execute when an action is called
4. **Parquet** is better than CSV for Spark workloads (columnar, compressed)
5. **Spark SQL** lets you use familiar SQL syntax on DataFrames

---

**Next**: [[02_banking_transformations.ipynb]] - Real banking transformations

In [27]:
# Clean up
spark.stop()
print("✅ Spark session stopped.")

✅ Spark session stopped.
